# Revisão - Indústria Alimentícia

## 1. Importação das bibliotecas e definições gerais

In [1]:
import os
import pandas as pd
import numpy as np
import urllib3
import importlib
import warnings

# Desabilita avisos de requisições HTTPS sem verificação de certificado
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Ignora especificamente o FutureWarning de "Downcasting" do pandas
warnings.filterwarnings(
    "ignore", 
    category=FutureWarning, 
    message="Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated"
)

import pandas as pd
import unicodedata # Serve para acessar e manipular as propriedades de caracteres

def clean_text(text):
    """
    Limpa uma string: remove acentos, espaços extras e converte para maiúsculo.

    Parâmetros:
        text (str): O texto a ser limpo.

    Retorna:
        str: O texto limpo, ou o valor original se for nulo.
    """
    # Se o valor for nulo (None, NaN), retorna como está para evitar erros.
    if pd.isna(text):
        return text

    # Garante que é string, remove espaços no início/fim e põe em maiúsculo.
    text = str(text).strip().upper()

    # Remove acentuação e caracteres especiais (ex: ç, á -> C, A).
    text = unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('ASCII')

    return text

## 2. Criação da base geral

### 2.1. Funções

#### Função auxiliar para analisar CNPJ e CPF

In [2]:
def CNPJAnalysis(df, cnpj_column='mv.num_cpf_cnpj'):
    """
    Analyzes CNPJ/CPF data in a DataFrame, classifying documents and counting digit lengths.
    
    Parameters:
        df (pd.DataFrame): DataFrame containing the documents to analyze
        cnpj_column (str): Name of the column containing CPF/CNPJ numbers (default: 'mv.num_cpf_cnpj')
    
    Returns:
        dict: A dictionary with the count of documents by digit length
        pd.DataFrame: The original DataFrame with added 'tipo' column (CNPJ/CPF/outro)
    """
    
    # 1. Classify documents by length
    df['status_v01'] = np.where(
        df[cnpj_column].str.len() == 14, 'CNPJ - Considerado para análise',
        np.where(
            df[cnpj_column].str.len() == 11, 'CPF - Desconsiderado para análise',
            'CPF - Desconsiderado para análise'
        )
    )
    
    # 2. Initialize counter for digit lengths (4-14)
    contagem = {length: 0 for length in range(4, 15)}
    total_documents = len(df)
    
    # 3. Count documents by cleaned digit length
    for doc in df[cnpj_column]:
        # Remove all non-digit characters
        cleaned_doc = ''.join(filter(str.isdigit, str(doc)))
        length = len(cleaned_doc)
        
        if length in contagem:
            contagem[length] += 1
    
    # 4. Verification and reporting
    total_counted = sum(contagem.values())
    
    if total_counted == total_documents:
        print("✅ Todos os documentos foram contabilizados!")
    else:
        print(f"⚠️ Atenção: {total_documents - total_counted} documentos não se enquadram nos tamanhos 4-14 dígitos")
    
    # 5. Detailed report
    print("\nDistribuição de dígitos:")
    for length, quantity in sorted(contagem.items()):
        if quantity > 0:  # Only show lengths that actually appear
            print(f'{quantity:>5} documentos com {length:>2} dígitos ({quantity/total_documents:.1%})')
    
    print(f"\nTotal contado: {total_counted} de {total_documents} documentos\nApenas serão considerados documentos com 14 dígitos (CNPJs)")
    
    return



#### Funções de download da base de dados online com CNPJ e localização


In [3]:
def download_ibama_ctf_data(repo_path):
    '''
    Automatiza o processo de download, processamento e consolidação de dados de
    Pessoas Jurídicas do Cadastro Técnico Federal (CTF) do IBAMA.

    O fluxo de trabalho da função é o seguinte:
    1. Cria as estruturas de pastas necessárias ('inputs/dadosIbamaCNPJ_Download'
       para arquivos brutos e 'inputs/dadosIbamaCNPJ_Consolidado' para o final).
    2. Pergunta ao usuário se os arquivos já foram baixados para pular a etapa de download.
    3. Se o download for necessário, itera por todas as Unidades da Federação (UFs),
       baixa o arquivo CSV correspondente e o salva na pasta de dados brutos.
    4. Cada arquivo baixado é processado individualmente para padronizar CNPJs e
       nomes de colunas.
    5. Ao final, todos os dados processados são consolidados em um único DataFrame.
    6. Colunas de texto ('Municipio', 'Razao Social') passam por uma limpeza final.
    7. O DataFrame consolidado é salvo como um único arquivo CSV na pasta de
       dados processados.

    Parâmetros:
        repo_path (str): Caminho para a pasta raiz do projeto. As subpastas de
                         dados serão criadas dentro deste caminho.

    Retorna:
        pandas.DataFrame: Um DataFrame contendo todos os dados consolidados e limpos.
        None: Retorna None se o processo falhar ou se nenhum arquivo for processado.
    '''
    
    # Define os caminhos para as pastas de dados brutos e processados
    raw_dir = os.path.join(repo_path, 'inputs', 'MaterialBaixado','PJ_UF')
    processed_dir = os.path.join(repo_path, 'inputs', 'MaterialBaixado')
    
    # Garante que as pastas de destino existam; se não, elas são criadas
    os.makedirs(raw_dir, exist_ok=True)
    os.makedirs(processed_dir, exist_ok=True)
    
    # Interage com o usuário para saber se a etapa de download pode ser pulada
    start = input('Você já tem os arquivos baixados? (s/n) ')
    
    # Se o usuário responder 's', o script carrega o arquivo já consolidado
    if start.lower() == 's':
        
        # Caminho para o arquivo final que deveria existir
        final_file_path = os.path.join(processed_dir, "PJ_BR.csv")
        print(f"Carregando arquivo consolidado de: {final_file_path}")
        
        # Lê o CSV consolidado, garantindo a tipagem correta de colunas-chave
        df_final = pd.read_csv(final_file_path,
                               dtype={'CNPJ': str,
                                      'Codigo da atividade': str,
                                      'Codigo da categoria': str},
                               keep_default_na=False)
               
        return df_final
    
    else:
        # Bloco de execução para o download e processamento dos dados
        print("Iniciando o download e processamento dos dados...")
        base_url = "http://dadosabertos.ibama.gov.br/dados/CTF/APP/"
        
        # Lista de todas as Unidades da Federação do Brasil
        ufs = [
            'AC', 'AL', 'AP', 'AM', 'BA', 'CE', 'DF', 'ES', 'GO',
            'MA', 'MT', 'MS', 'MG', 'PA', 'PB', 'PR', 'PE', 'PI',
            'RJ', 'RN', 'RS', 'RO', 'RR', 'SC', 'SP', 'SE', 'TO'
        ]
        
        # Lista vazia para armazenar os DataFrames de cada estado após o processamento
        pessoas_juridicas_br = []
        
        # Itera sobre cada UF para baixar e processar os dados
        for uf in ufs:
            try:
                # Constrói a URL completa para o arquivo CSV do estado atual
                url = f"{base_url}{uf}/pessoasJuridicas.csv"
                print(f"Baixando {uf}...")
                
                # Faz a requisição GET para baixar o arquivo
                # timeout=30: define um tempo limite de 30 segundos
                # verify=False: ignora erros de certificado SSL (usar com cautela)
                response = requests.get(url, verify=False, timeout=30) 
                
                # Verifica se a requisição foi bem-sucedida (status code 2xx)
                response.raise_for_status() 
                
                # Define o caminho completo para salvar o arquivo bruto baixado
                output_path = os.path.join(raw_dir, f"pessoasJuridicas_{uf}.csv")
                
                # Abre o arquivo em modo de escrita binária ('wb') e salva o conteúdo
                with open(output_path, 'wb') as f:
                    f.write(response.content)
                
                # Inicia o processamento do arquivo recém-baixado
                try:
                    # Carrega o arquivo CSV em um DataFrame, especificando o delimitador e os tipos de dados
                    df = pd.read_csv(output_path,
                                     delimiter=';',
                                     dtype={'CNPJ': str,
                                            'Codigo da atividade': str,
                                            'Codigo da categoria': str},
                                     keep_default_na=False)
                    
                    # Padroniza a coluna de CNPJ para ter sempre 14 dígitos, com zeros à esquerda
                    df['CNPJ'] = df['CNPJ'].str.zfill(14) 
                    
                    # Limpa e padroniza os nomes das colunas usando a função externa clean_text
                    df.columns = df.columns.map(clean_text)
                    
                    # Adiciona o DataFrame do estado, já limpo, à lista de consolidação
                    pessoas_juridicas_br.append(df)
                    
                # Captura e informa erros que possam ocorrer durante o processamento do arquivo
                except Exception as e:
                    print(f"Erro ao processar {uf}: {e}")
                    
            # Captura e informa erros relacionados ao download (ex: rede, URL inválida)
            except requests.exceptions.RequestException as e:
                print(f"Erro ao baixar {uf}: {e}")
        
        # Após o loop, verifica se algum dado foi processado antes de consolidar
        if pessoas_juridicas_br:
            print("\nConsolidando todos os dados...")
            # Junta todos os DataFrames da lista em um único DataFrame
            df_final = pd.concat(pessoas_juridicas_br, ignore_index=True)
            
            # Aplica a função de limpeza nas células das colunas 'Municipio' e 'Razao Social'
            df_final['MUNICIPIO'] = df_final['MUNICIPIO'].apply(clean_text)
            df_final['RAZAO SOCIAL'] = df_final['RAZAO SOCIAL'].apply(clean_text)
            df_final['MUNICIPIO'] = df_final['MUNICIPIO'].str.replace(
                r"TRAJANO DE MORAIS", "TRAJANO DE MORAES", regex=True
                )
            
            # Extrair os anos de inicio e fim das datas
            # ANO_INICIO
            df_final['ANO_INICIO'] = (
                df_final['DATA DE INICIO DA ATIVIDADE']
                .fillna('0000')                    # trata valores nulos
                .replace('', '0000')               # trata strings vazias
                .str[-4:]                          # pega os 4 últimos caracteres
                .astype(int)                       # converte para inteiro
            )
            
            # ANO_FIM
            df_final['ANO_FIM'] = (
                df_final['DATA DE TERMINO DA ATIVIDADE']
                .fillna('0000')                    # trata valores nulos
                .replace('', '0000')               # trata strings vazias
                .str[-4:]
                .astype(int)
            )
            #Onde a data de inicio e fim de atividade for vazio, inserir np.nan
   
            # Define o caminho e nome do arquivo final e o salva em formato CSV
            output_file = os.path.join(processed_dir, "PJ_BR.csv")
            df_final.to_csv(output_file, index=False, encoding='utf-8')
            print("Dados baixados e processados com sucesso!")
            
            # Retorna o DataFrame final
            return df_final
            
    # Se a lista 'pessoas_juridicas_br' estiver vazia, retorna None
    print("Nenhum dado foi processado.")
    return None

#### Funções de importação de base de dados confidencial encaminhada pelo IBAMA

Processa dados de produção do IBAMA a partir de arquivos XLSX, limpando e consolidando os dados. Retorna:
- pd.DataFrame: DataFrame com os dados processados
- str: Caminho do arquivo CSV salvo

In [4]:
def ibama_production_data_17_23(repo_path):
    """
    """

    # Define os caminhos para as pastas de dados brutos e processados
    raw_dir = os.path.join(repo_path, 'inputs','DadosProduçãoIndustrial')
    processed_dir = os.path.join(repo_path, 'inputs','DadosProduçãoIndustrial')
    
    # Garante que as pastas de destino existam; se não, elas são criadas
    os.makedirs(raw_dir, exist_ok=True)
    os.makedirs(processed_dir, exist_ok=True)
    
    try:
        # Caminho completo do arquivo
        file_path = os.path.join(raw_dir, 'DadosProduçãoBruto.xlsx')
        
        # Lê as duas abas do Excel
        aba1 = pd.read_excel(file_path, sheet_name=0, dtype={'mv.num_cpf_cnpj': str}) # 0 para a primeira aba
        aba2 = pd.read_excel(file_path, sheet_name=1, dtype={'mv.num_cpf_cnpj': str}) # 1 para segunda aba
                
        # Combina as abas, deleta as linhas duplicadas
        df_ibama_prod = pd.concat([aba1, aba2], ignore_index=True)
        
        
        # Padronização dos dados com a função clean_text
        df_ibama_clean = df_ibama_prod.copy()
        df_ibama_clean['mv.nom_municipio'] = df_ibama_clean['mv.nom_municipio'].apply(clean_text)
        df_ibama_clean['mv.nom_pessoa'] = df_ibama_clean['mv.nom_pessoa'].apply(clean_text)
        df_ibama_clean['mv.nom_municipio'] = df_ibama_clean['mv.nom_municipio'].str.replace(
            r"SANT'? ?ANA DO LIVRAMENTO", "SANTANA DO LIVRAMENTO", regex=True
            )
        df_ibama_clean['mv.nom_municipio'] = df_ibama_clean['mv.nom_municipio'].str.replace(
            r"PRESIDENTE CASTELLO BRANCO", "PRESIDENTE CASTELO BRANCO", regex=True
            )
              
        # Salva o resultado
        output_file = os.path.join(processed_dir, 'DadosProduçãoTratado_17_23.csv')
        df_ibama_clean.to_csv(output_file, index=False, encoding='utf-8') #Remove índices
        
        return df_ibama_clean
        
    # Caso dê erro, informa ao usuário
    except FileNotFoundError:
        raise FileNotFoundError(f"Arquivo RAPP.xlsx não encontrado em {raw_dir}")
    except Exception as e:
        raise Exception(f"Erro ao processar dados: {str(e)}")

#### Funções que mescla produção com coordenadas

In [5]:
def merge_cnpj_prod(cnpj,prod):
    '''
    
    Processa dados de atividade por CNPJ e Município para obter coordenada,
    código de atividade e código de categoria; Remove os que são CPF   
    
    Parâmetros: 
        - cnpj, prod = Base de dados de CNPJ e de Produção do ibama, e baixado pelas funções:
            -download_ibama_ctf_data
            -ibama_production_data
            
     Retorna:
         pd.DataFrame: DataFrame com os dados concatenados
         
    '''
    
    # Uma mesma indústria com o mesmo cnpj e lat lon produz produtos diferentes
    # Ou seja, multiplos codigo de categoria e codigo de atividade
    # Por isso, fiz um groupby onde os códigos de categoria e atividade viram uma lista para um mesmo CNPJ
    # acabei não utilizando estes códigos, mas mantive por segurança
    cnpj = cnpj.groupby(['CNPJ', 'MUNICIPIO', 'LATITUDE', 'LONGITUDE','ESTADO','SITUACAO CADASTRAL']).agg({
        'CODIGO DA CATEGORIA': list,
        'CODIGO DA ATIVIDADE': list,
        'ANO_INICIO': list,
        'ANO_FIM': list
    }).reset_index()
    
    
    # Fazer o merge com o df_ibama (sem duplicar linhas)
    df_ibama_completo = pd.merge(
        left=cnpj, #tabela unida a esqueda
        right=prod, #tabela unida a direita
        left_on=['CNPJ', 'MUNICIPIO'], #chaves da tabela a esqueda
        right_on=['mv.num_cpf_cnpj', 'mv.nom_municipio'], #chaves da tabela a direita
        how='right', # todas as linhas da tabela da direita (prod) serão mantidas.
        indicator=True #informa a condição da mesclagem (right_only - não estava em CNPJ)
    )
    
       
    return df_ibama_completo

#### Função que agrupa e agrega os valores

In [6]:
def agrupar_e_somar_dados(df: pd.DataFrame) -> pd.DataFrame:
    """
    Agrupa um DataFrame por um conjunto de colunas-chave e agrega os valores.

    As colunas numéricas (não presentes nas chaves de agrupamento) são somadas.
    As colunas não numéricas são preenchidas com o primeiro valor do grupo.

    Args:
        df (pd.DataFrame): O DataFrame de entrada para ser processado.

    Returns:
        pd.DataFrame: Um novo DataFrame com os dados agrupados e agregados.
    """
    # 1. Define as colunas que serão a "chave" para o agrupamento
    colunas_agrupamento = [
        'mv.num_cpf_cnpj', 
        'mv.nom_pessoa',
        'mv.nom_municipio',
        'num_ano',
        'cod_produto',
        'unidade_medida',
        'sig_unidmed'
    ]

    print(f"Número de linhas antes do agrupamento: {len(df)}")

    # 2. Cria dinamicamente as regras de agregação
    # O objetivo é dizer ao pandas o que fazer com cada coluna que NÃO está no agrupamento.
    regras_agregacao = {}
    colunas_para_agregar = [col for col in df.columns if col not in colunas_agrupamento]

    for coluna in colunas_para_agregar:
        # Verifica se a coluna contém dados numéricos (int, float, etc.)
        if pd.api.types.is_numeric_dtype(df[coluna]):
            regras_agregacao[coluna] = 'sum'  # Se for numérica, some
        else:
            regras_agregacao[coluna] = 'first' # Se não for, pegue o primeiro valor

    # 3. Executa o agrupamento e a agregação
    # .groupby() cria os grupos
    # .agg() aplica as regras que definimos
    # .reset_index() transforma as colunas de agrupamento de volta em colunas normais
    df_agrupado = df.groupby(colunas_agrupamento).agg(regras_agregacao).reset_index()

    print(f"Número de linhas após o agrupamento: {len(df_agrupado)}")
    
    return df_agrupado

### 2.2. Código

In [7]:
# Caminho da pasta do projeto
# Dentro do repo_path deve-se ter as pastas
# repo_path
#   |--figures
#   |--inputs
#   |--outputs
#       |--log
#   |--scripts

repo_path = os.path.dirname(os.getcwd())

# Faz o downloado da base de dados com CNPJ + Coordenadas
df_ibama_cnpj = download_ibama_ctf_data(repo_path)

# Importa DF com Dados de produção com CNPJ + Código de Produto + Produção
df_ibama_prod = ibama_production_data_17_23(repo_path)

#Garante que não existam linhas duplicadas com base em: 'mv.num_cpf_cnpj', 'mv.nom_pessoa',
#'mv.nom_municipio','num_ano','cod_produto','unidade_medida','sig_unidmed'
df_ibama_prod = agrupar_e_somar_dados(df_ibama_prod)

# DF com Produção + Código de produto + Coordenadas
#mesclar p obter coordenadas e código de atividade
df_ibama = merge_cnpj_prod(df_ibama_cnpj,df_ibama_prod) 

#contabiliza quantos são CPF (11 dígitos) e quantos são CNPJ (14 dígitos)
CNPJAnalysis(df_ibama)

#Exporta log com dados desconsiderados
df_ibama.to_excel(os.path.join(repo_path,'outputs','log','log_v01_remocaoCPF.xlsx'))

#Remover CPFs (desconsiderável)
df_ibama = df_ibama[df_ibama['mv.num_cpf_cnpj'].str.len() == 14]

Carregando arquivo consolidado de: c:\Users\glima\OneDrive\Documentos\Mestrado_GitHub\006.2026 - Revisão TCC\inputs\MaterialBaixado\PJ_BR.csv
Número de linhas antes do agrupamento: 572434
Número de linhas após o agrupamento: 542135
✅ Todos os documentos foram contabilizados!

Distribuição de dígitos:
 1973 documentos com 11 dígitos (0.4%)
540162 documentos com 14 dígitos (99.6%)

Total contado: 542135 de 542135 documentos
Apenas serão considerados documentos com 14 dígitos (CNPJs)


## 3. Base com NFR + Código de Produto + Produção

### 3.1. Funções

Importa e limpa a tabela de códigos de produtos do IBGE (PRODLIST) APENAS PRODUTOS ALIMENTÍCIOS (escopo inicial do TCC).
- Verificar online: https://servicos.ibama.gov.br/ctfcd/manual/html/lista_produtos.htm
- Baixar excel: https://www.ibge.gov.br/estatisticas/metodos-e-classificacoes/classificacoes-e-listas-estatisticas/9153-lista-de-produtos-da-industria.html
   
A função lê um arquivo Excel, remove cabeçalhos repetidos e linhas indesejadas, e retorna um DataFrame pronto para uso. Retorna:
- pd.DataFrame: Tabela de códigos de produtos limpa.  

In [8]:
def import_treat_export_food_code(repo_path):
    raw_dir = os.path.join(repo_path, 'inputs','MaterialBaixado')
    processed_dir = os.path.join(repo_path, 'outputs')
    
    # Garante que as pastas de destino existam; se não, elas são criadas
    os.makedirs(raw_dir, exist_ok=True)
    os.makedirs(processed_dir, exist_ok=True)
    
    # Caminho do arquivo
    xlsx_path = os.path.join(raw_dir,'CódigosProdutosIBGE.xlsx')

    # Lê o CSV com header na linha 1 (índice 1)
    cod_produto = pd.read_excel(xlsx_path, header=2, dtype={'PRODLIST': str})
    # Remove linhas com 'PRODLIST' ou NaN na coluna 'PRODLIST'
    cod_produto = cod_produto[~cod_produto['PRODLIST'].isin(['PRODLIST'])]
    cod_produto = cod_produto[~cod_produto['PRODLIST'].astype(str).str.startswith('CNAE')]
    cod_produto = cod_produto.dropna(subset=['PRODLIST'])

    # Reinicia o índice
    cod_produto.reset_index(drop=True, inplace=True)
       
    return cod_produto

Conecta a base de dados do IBAMA ao fator de emissão adotado manualmente

In [9]:
def conecta_ibama_ef(df_ibama, df_ef, df_conector):
    # Garantir que os códigos estejam no mesmo tipo
    df_conector[['PRODLIST', 'NFR', 'Table']] = df_conector[['PRODLIST', 'NFR', 'Table']].astype(str)
    df_ibama['cod_produto'] = df_ibama['cod_produto'].astype(str)
    
    # Mesclar df_ibama com o conector via código do produto
    df_merged = df_ibama.merge(
        df_conector[['PRODLIST', 'NFR', 'Table']],
        left_on='cod_produto',
        right_on='PRODLIST',
        how='left'
    )

    # Realizar o merge com a base ef (EEA)
    df_final = df_merged.merge(
        df_ef,
        left_on=['NFR', 'Table'],
        right_on=['NFR', 'Table'],
        how='left'
    )

    # Remover coluna auxiliar
    #df_final = df_final.drop(columns=['PRODLIST'])

    return df_final

### 3.2. Código

Aqui eu importo a base de dados com os códigos de produto do IBGE, faço uma limpeza inicial e exporto para a pasta de outputs para classificar manualmente quais códigos se enquadram no item 2.h.2 do NFR (Alimentos e Bebidas)

In [10]:
#Base de dados com todos os Códigos de Produto
cod_produto= import_treat_export_food_code(repo_path)

#vou filtrar os códigos de produto de interesse e exportar (itens que se enquadram no 2.h.2 no nfr)
cod_produto_interesse = cod_produto[
    cod_produto['PRODLIST'].astype(str).str.startswith(('10', '11'))
]

#exportei para classificar manualmente
cod_produto_interesse.to_excel(os.path.join(repo_path, 'outputs','1_CodProdutoClassificadoNFR.xlsx'), index=False)

Após classificar os códigos de produto em NFR, exportar manualmente para a pasta inputs. Os classificados como "Não" serão desconsiderados.

In [11]:
#Importei material gerado manualmente
#VERIFICACAO
CodProdutoClassificadoNFR = pd.read_excel(os.path.join(repo_path,'inputs','MaterialGeradoManualmente','1_CodProdutoClassificadoNFR.xlsx'),
                                          dtype={'PRODLIST': str})

#filtrei os alimentos que tem emissões a serem consideradas
CodProdutoClassificadoNFR = CodProdutoClassificadoNFR[CodProdutoClassificadoNFR['EmissaoNMCOV_NFR'] != 'Não']

# Base de dados dos fatores de emissão tier 2
eea_ef = pd.read_csv(os.path.join(repo_path, 'inputs','MaterialBaixado', 'EF_tier2.csv'))

# Conexão das bases de dados de apenas os classificados como emissores de NMCOV
df_ibama_EF = conecta_ibama_ef(df_ibama,eea_ef,CodProdutoClassificadoNFR)

#log que indica os removidos por não serem alimentos emissores de COV
# 1. Crie a coluna com o valor padrão para todas as linhas
df_ibama_EF['status_v02'] = 'Dado filtrado'

# 2. Use .loc para encontrar as linhas que batem com a condição e mude o valor apenas nelas
df_ibama_EF.loc[df_ibama_EF['NFR'] == '2.H.2', 'status_v02'] = 'Alimento emissor de COV - Dado considerado'

#Exporta log com dados desconsiderados
df_ibama_EF.to_excel(os.path.join(repo_path,'outputs','log','log_v02_apenasAlimentosEmissoresMantidos.xlsx'))

#Filtrar apenas os produtos com emissão
df_ibama_EF = df_ibama_EF[df_ibama_EF['Table'].notna()]

## 4. Ajuste 01: Indústrias que reportaram mais de uma unidade de medida

### 4.1 Função

### 4.2 Código

Primeiro, realiza-se as correções automáticas. Ex: Lv --> L; KWh --> Kg (para o setor alimentício, estes erros são facilmente cometidos)

O objetivo desta seção é encontrar os grupos (CNPJ, Município, Produto) que possuem mais de uma unidade de medida declarada ao longo do tempo e corrigi-las para uma unidade coerente.

In [12]:
# Para cada combinação única de [CNPJ, MUNICIPIO, cod_produto], calcula-se o total de registros.
# A função 'transform' aplica essa contagem a todas as linhas do grupo original,
# criando a coluna 'contagem_total_grupo' para uso posterior na análise.
df_ibama_EF['contagem_total_grupo'] = df_ibama_EF.groupby(
    ['CNPJ', 'MUNICIPIO', 'cod_produto']
)['cod_produto'].transform('count')

# Agrupa-se novamente e conta-se a frequência de cada 'unidade_medida' dentro do mesmo grupo.
# O resultado é um DataFrame ('df_valid_final') que mostra, por exemplo:
# CNPJ A, Produto X -> 'tonelada': 10 vezes, 'unidade': 2 vezes.
df_valid_final = df_ibama_EF.groupby(
    ['CNPJ', 'MUNICIPIO', 'cod_produto', 'contagem_total_grupo']
)['unidade_medida'].value_counts().reset_index(name='contagem_unidade')

# Um grupo é inconsistente se possuir mais de um tipo de unidade de medida.
# A lógica aqui é identificar no 'df_valid_final' os grupos que aparecem mais de uma vez.
grupos_inconsistentes = df_valid_final[df_valid_final.duplicated(
    subset=['CNPJ', 'MUNICIPIO', 'cod_produto'], keep=False
)]

# Gera uma lista limpa e sem duplicatas dos grupos que precisam ser corrigidos manualmente.
df_para_verificar = grupos_inconsistentes[['CNPJ', 'MUNICIPIO', 'cod_produto']] \
    .drop_duplicates() \
    .reset_index(drop=True)

# Exporta a lista de grupos inconsistentes para um arquivo Excel.
# Este arquivo servirá como um guia para a verificação manual.
df_para_verificar.to_excel(os.path.join(repo_path,
                                        'outputs',
                                        'Auxiliar_dfUnidadesInconsistentes_VerifManual.xlsx'),
                           index=False)


Nesta etapa, o DataFrame principal é preparado e exportado para que as correções sejam feitas manualmente em um software de planilha. A correção manual consiste e inserir na coluna 'sig_unidmed_novo' a unidade adequada, de forma a deixar coerente com as demais. Deve ser inserido a categoria geral do alimento, para facilitar as análises posteriores. Ex: Pão; Carnes; Etc..

In [13]:

# ── ETAPA 1: CORREÇÕES AUTOMÁTICAS DE UNIDADES ──────────────────────────────
sinonimos_und = {
    'KG/100KG': 'KG',
    'L/100 L':  'L',
    'T/M³':     'TON',
    'LV':       'L',
    'LT':       'L',
    'KG/HA':    'KG',
    'G/KG':     'G',
    'L/M3':     'L',
    'KM3':      'M3',
    'G/M3':     'G',
    'ML/TON':   'ML',
    'ML/100 L': 'ML',
    'ML/L':     'ML',
    'T/ST':     'TON',
}

unidades_incoerentes = {
    'UN', 'CX', 'PC', 'ANIMAL', 'DZ', 'PÇ',
    'MWH', 'KWH', 'BA', 'FR', 'CT', 'MI', 'UR', 'SAMP'
}

# Inicializa colunas
df_ibama_EF['status_v03'] = 'Unidade mantida conforme dado original'
df_ibama_EF['sig_unidmed_novo'] = df_ibama_EF['sig_unidmed'].str.strip().str.upper()

# Aplica sinônimos
mask_sin = df_ibama_EF['sig_unidmed_novo'].isin(sinonimos_und)
df_ibama_EF.loc[mask_sin, 'sig_unidmed_novo'] = (
    df_ibama_EF.loc[mask_sin, 'sig_unidmed_novo'].map(sinonimos_und)
)
df_ibama_EF.loc[mask_sin, 'status_v03'] = (
    'Unidade alterada automaticamente - Considerado erro de digitação'
)

# Marca incoerentes
mask_inc = df_ibama_EF['sig_unidmed_novo'].isin(unidades_incoerentes)
df_ibama_EF.loc[mask_inc, 'status_v03'] = 'Unidade incoerente - Desconsiderado'

print(f"Auto-correções aplicadas : {mask_sin.sum()} registros")
print(f"Unidades incoerentes     : {mask_inc.sum()} registros")

# ── EXPORTA COMBINAÇÕES ÚNICAS PARA PREENCHIMENTO MANUAL DOS FATORES ─────────
# Exclui incoerentes; mantém só o que precisa de fator de conversão
colunas_export = ['cod_produto', 'nom_produto', 'sig_unidmed_novo', 'Unit']
colunas_export = [c for c in colunas_export if c in df_ibama_EF.columns]

df_combinacoes = (
    df_ibama_EF[~mask_inc][colunas_export]
    .groupby(colunas_export)
    .size()
    .reset_index(name='contagem')
    .sort_values(colunas_export)
    .reset_index(drop=True)
)

df_combinacoes = df_combinacoes.rename(columns={
    'cod_produto':    'cod_produto_fc',
    'nom_produto':    'nom_produto_fc',
    'sig_unidmed_novo': 'sig_unidmed_fc',
})

df_combinacoes['Observação'] = ''
df_combinacoes['cod_produto_fc'] = df_combinacoes['cod_produto_fc'].astype(str)

# Fatores universais por unidade (independente do produto)
fatores_universais_Mg = {'TON': 1.0, 'KG': 1e-3, 'G': 1e-6}
fatores_universais_hL = {'HL': 1.0, 'L': 0.01, 'M3': 10.0, 'ML': 1e-5}

def _get_fator_universal(row):
    und  = row['sig_unidmed_fc']
    unit = str(row['Unit'])
    
    # Álcool: deixa em branco (depende do teor alcoólico do produto)
    if '/hl alcohol' in unit:
        return ''
    
    if '/Mg' in unit:
        return fatores_universais_Mg.get(und, '')  # '' se precisar de densidade
    
    if '/hl' in unit:
        return fatores_universais_hL.get(und, '')  # '' se precisar de densidade
    
    return ''

df_combinacoes['fatorConversao'] = df_combinacoes.apply(_get_fator_universal, axis=1)
df_combinacoes['categoria'] = ''

df_combinacoes.to_excel(
    os.path.join(repo_path, 'outputs', '2_UnidadesFatorConversão.xlsx')
)

print(f"\nExportado: {len(df_combinacoes)} combinações únicas para outputs/UnidadesFatorConversão.csv")
print("→ Preencha fatorConversao, copie para inputs/MaterialGeradoManualmente/ e rode novamente. Preencha coluna 'categoria' para preencher a categoria geral de emissão")


Auto-correções aplicadas : 199 registros
Unidades incoerentes     : 798 registros

Exportado: 139 combinações únicas para outputs/UnidadesFatorConversão.csv
→ Preencha fatorConversao, copie para inputs/MaterialGeradoManualmente/ e rode novamente. Preencha coluna 'categoria' para preencher a categoria geral de emissão


In [14]:
# ── CARREGA TABELA DE FATORES DE CONVERSÃO (fonte única) ────────────────────
df_fatores = pd.read_excel(
    os.path.join(repo_path, 'inputs', 'MaterialGeradoManualmente', '2_UnidadesFatorConversão.xlsx'),
    dtype={'cod_produto_fc': str}
)

cores_categoria = {
    "Açucar":                       "beige",
    "Café":                         "brown",
    "Gorduras sólidas":             "yellow",
    "Bolos, biscoitos e cereais":   "grey",
    "Carne e Ração Animal":         "salmon",
    "Vinho":                        "purple",
    "Pão":                          "pink",
    "Cerveja e malte":              "goldenrod",
    "Destilados":                   "lightblue",
}

df_fatores["food_color"] = df_fatores["categoria"].map(cores_categoria)
print("Linhas sem cor:", df_fatores["food_color"].isna().sum())

# Normaliza unidade do CSV para bater com sig_unidmed_novo (UPPER + strip)
df_fatores['_und_norm'] = df_fatores['sig_unidmed_fc'].str.strip().str.upper()

# Lookup: (cod_produto, unidade_normalizada) → fatorConversao
# Considera apenas fatores > 0 (fator 0 = incoerente no CSV antigo)
lookup_fator = (
    df_fatores[df_fatores['fatorConversao'] > 0]
    .set_index(['cod_produto_fc', '_und_norm'])['fatorConversao']
    .to_dict()
)

Linhas sem cor: 0


In [15]:
# ── ETAPA 2: RESOLUÇÃO DE GRUPOS COM MÚLTIPLAS UNIDADES ─────────────────────
group_cols = ['CNPJ', 'MUNICIPIO', 'cod_produto']

# Busca fator de conversão no CSV por (cod_produto, sig_unidmed_novo)
df_ibama_EF['_fator_und'] = df_ibama_EF.apply(
    lambda r: lookup_fator.get((r['cod_produto'], r['sig_unidmed_novo'])),
    axis=1
)

df_ibama_EF['_fator_und']=df_ibama_EF['_fator_und'].fillna(0)  # Marca como 0 os que não têm fator (incoerentes ou sem dados)

df_ibama_EF['_qtd_convertida'] = (
    df_ibama_EF['qtd_produzida'].astype(float) * df_ibama_EF['_fator_und']
)



In [16]:
# ── CLASSIFICAÇÃO PRÉVIA: posição no Q90 do setor ───────────────────────────
# 1. Traz a 'categoria' (preenchida no CSV de fatores) para o df principal
lookup_categoria = (
    df_fatores.dropna(subset=['categoria'])
    .drop_duplicates('cod_produto_fc')
    .set_index('cod_produto_fc')['categoria']
    .to_dict()
)
df_ibama_EF['categoria'] = df_ibama_EF['cod_produto'].map(lookup_categoria)

# 2. Q90 por setor, calculado só sobre linhas que têm valor convertido real
percentil = 0.90
tem_valor = df_ibama_EF['_fator_und'] > 0

q90_por_cat = (
    df_ibama_EF.loc[tem_valor]
    .groupby('categoria')['_qtd_convertida']
    .quantile(percentil)
)
df_ibama_EF['q90_setor_prev'] = df_ibama_EF['categoria'].map(q90_por_cat)

# 3. Classifica cada linha
df_ibama_EF['status_q90_prev'] = np.select(
    [
        df_ibama_EF['categoria'].isna(),
        ~tem_valor,
        df_ibama_EF['_qtd_convertida'] > df_ibama_EF['q90_setor_prev'],
    ],
    [
        'Sem categoria',
        'Sem valor convertido',
        '> Q90 do setor',
    ],
    default='<= Q90 do setor'
)

print(df_ibama_EF['status_q90_prev'].value_counts())

status_q90_prev
<= Q90 do setor         12716
> Q90 do setor           1417
Sem valor convertido      798
Name: count, dtype: int64


In [17]:
# Identifica grupos com múltiplas unidades (excluindo incoerentes)
n_und_grupo = (
    df_ibama_EF[~mask_inc]
    .groupby(group_cols)['sig_unidmed_novo']
    .nunique()
)
grupos_multi_set = set(n_und_grupo[n_und_grupo > 1].index)

mask_multi = (
    pd.MultiIndex.from_frame(df_ibama_EF[group_cols]).isin(grupos_multi_set)
    & ~mask_inc
)
idx_multi = df_ibama_EF[mask_multi].index

print(f"\nGrupos com múltiplas unidades (após auto-correção): {len(grupos_multi_set)}")


Grupos com múltiplas unidades (após auto-correção): 606


In [18]:

# Verifica ordem de grandeza após conversão
if len(idx_multi) > 0:
    mediana_grupo = (
        df_ibama_EF.loc[idx_multi]
        .groupby(group_cols)['_qtd_convertida']
        .transform(lambda x: x.median(skipna=True))
    )
    ratio = df_ibama_EF.loc[idx_multi, '_qtd_convertida'] / mediana_grupo

    fator_ok      = df_ibama_EF.loc[idx_multi, '_fator_und'].notna()
    dentro_escala = (ratio >= 0.1) & (ratio <= 10)

    idx_ok        = idx_multi[fator_ok & dentro_escala]
    idx_suspeito  = idx_multi[fator_ok & ~dentro_escala]
    idx_sem_fator = idx_multi[~fator_ok]

    df_ibama_EF.loc[idx_ok,        'status_v03'] = 'Unidade convertida automaticamente para unidade-alvo'
    df_ibama_EF.loc[idx_suspeito,  'status_v03'] = 'Conversão suspeita - Verificar escala'
    df_ibama_EF.loc[idx_sem_fator, 'status_v03'] = 'Unidade sem fator no CSV - Adicionar entrada'

    print(f"  Convertidos OK        : {len(idx_ok)}")
    print(f"  Suspeitos (escala)    : {len(idx_suspeito)}")
    print(f"  Sem fator no CSV      : {len(idx_sem_fator)}")

df_ibama_EF = df_ibama_EF.rename(columns={'_qtd_convertida': 'qtd_convertida_prev'})
df_ibama_EF = df_ibama_EF.drop(columns=['_fator_und'])

# Resumo e exportação
print(f"\nResumo status_v03:")
print(df_ibama_EF['status_v03'].value_counts())

n_manual = df_ibama_EF['status_v03'].isin([
    'Conversão suspeita - Verificar escala',
    'Unidade sem fator no CSV - Adicionar entrada'
]).sum()
print(f"\n→ {n_manual} registros restam para verificação manual")

  Convertidos OK        : 2675
  Suspeitos (escala)    : 525
  Sem fator no CSV      : 0

Resumo status_v03:
status_v03
Unidade mantida conforme dado original                              10788
Unidade convertida automaticamente para unidade-alvo                 2675
Unidade incoerente - Desconsiderado                                   798
Conversão suspeita - Verificar escala                                 525
Unidade alterada automaticamente - Considerado erro de digitação      145
Name: count, dtype: int64

→ 525 registros restam para verificação manual


### Retirada de séries históricas inconsistentes

In [19]:
# ── FILTRO DE SÉRIE HISTÓRICA CONSISTENTE ───────────────────────────────────
# Extraído da Etapa 1 do tratamento_outliers_v3 (depende só dos anos).
# Conta anos apenas em linhas com unidade coerente (as incoerentes serão descartadas).
group_cols = ['CNPJ', 'MUNICIPIO', 'cod_produto']
year_col   = 'num_ano'

def _historico_suficiente_anos(serie_anos):
    anos = sorted(set(serie_anos))
    if len(anos) >= 5:
        return True
    if len(anos) >= 3:
        for i in range(len(anos) - 2):
            if anos[i+1] - anos[i] == 1 and anos[i+2] - anos[i+1] == 1:
                return True
    return False

coerente = df_ibama_EF['status_v03'] != 'Unidade incoerente - Desconsiderado'

flag_por_grupo = (
    df_ibama_EF[coerente]
    .groupby(group_cols)[year_col]
    .agg(_historico_suficiente_anos)
    .rename('_hist_ok')
)

df_ibama_EF = df_ibama_EF.merge(flag_por_grupo, on=group_cols, how='left')
df_ibama_EF['_hist_ok'] = df_ibama_EF['_hist_ok'].fillna(False)
df_ibama_EF['status_serie'] = np.where(
    df_ibama_EF['_hist_ok'], 'Série Histórica Apta', 'Histórico Insuficiente'
)
df_ibama_EF = df_ibama_EF.drop(columns='_hist_ok')

print("--- Filtro de Série Histórica ---")
print(df_ibama_EF['status_serie'].value_counts())



--- Filtro de Série Histórica ---
status_serie
Série Histórica Apta      11954
Histórico Insuficiente     2977
Name: count, dtype: int64


#### Correção manual abaixo

In [20]:
# ── AUXILIAR: grupos com unidade inconsistente E algum valor > Q90 do setor ──
status_verificar = [
    'Conversão suspeita - Verificar escala',
    'Unidade sem fator no CSV - Adicionar entrada',
]

# 1. Grupos (CNPJ+MUN+produto) que têm PELO MENOS UM valor acima do Q90 do setor
grupos_q90_set = set(
    df_ibama_EF.loc[
        df_ibama_EF['status_q90_prev'] == '> Q90 do setor',
        group_cols
    ].itertuples(index=False, name=None)
)
mask_grupo_q90 = pd.MultiIndex.from_frame(df_ibama_EF[group_cols]).isin(grupos_q90_set)

# 2. Linhas a verificar: unidade inconsistente + série apta + grupo encosta no Q90
mask_verificar = (
    df_ibama_EF['status_v03'].isin(status_verificar)
    & (df_ibama_EF['status_serie'] == 'Série Histórica Apta')
    & mask_grupo_q90
)

cols_aux = ['CNPJ', 'MUNICIPIO', 'cod_produto', 'nom_produto',
            'sig_unidmed', 'sig_unidmed_novo', 'Unit',
            'qtd_convertida_prev', 'status_q90_prev', 'status_v03']
cols_aux = [c for c in cols_aux if c in df_ibama_EF.columns]

df_aux_verificar = (
    df_ibama_EF.loc[mask_verificar, cols_aux]
    .drop_duplicates()
    .sort_values(['status_v03', 'cod_produto', 'CNPJ'])
    .reset_index(drop=True)
)

df_aux_verificar.to_excel(
    os.path.join(repo_path, 'outputs', '3A_vefirManualUnidadesInconsistentes.xlsx'),
    index=False
)

print(f"Auxiliar exportado: {len(df_aux_verificar)} linhas para verificação manual")
print(f"  CNPJs distintos:  {df_aux_verificar['CNPJ'].nunique()}")
print(f"  Grupos distintos: {df_aux_verificar[group_cols].drop_duplicates().shape[0]}")

df_ibama_EF.to_excel(
    os.path.join(repo_path, 'outputs', '3_vefirManualUnidadesInconsistentes.xlsx'),
    index=False
)

Auxiliar exportado: 235 linhas para verificação manual
  CNPJs distintos:  141
  Grupos distintos: 149


Após a correção manual, o arquivo é reimportado para o script e uma cópia é salva como log para garantir a rastreabilidade do processo.

In [21]:
# Lê o arquivo Excel que foi modificado manualmente, contendo as unidades padronizadas.
# Este arquivo deve ser previamente salvo na pasta de inputs manuais.
df_ibama_EF_und = pd.read_excel(os.path.join(repo_path,
                                             'inputs',
                                             'MaterialGeradoManualmente',
                                             '3_vefirManualUnidadesInconsistentes.xlsx'),
                                dtype={'cod_produto': 'string','CNPJ':'string'})

# Salva uma cópia do DataFrame já corrigido no diretório de log.
# Esta ação preserva o estado dos dados após a correção manual
df_ibama_EF_und.to_excel(os.path.join(repo_path,
                                      'outputs',
                                      'log',
                                      'log_v03_AnaliseManualUndInconsistente.xlsx'))

## 5. Análise 02: Alteração da escala de produção

### 5.1 Código

Nessa etapa, buscou-se a correção de escalas de produção, dividindo ou multiplicando o valor de produção por múltimos de 10, tornando os valores coerentes. Esta etapa foi realizada concomitantemente com a exportação da comparação com o PIA, buscando-se dados coerentes com a base de dados indicadora.

Dessa forma, primeiro exportou-se uma base de dados com a produção classificada por ser maior ou menor que o Q90 do grupo, e vou analisar os CNPJs classificados como sim. Em seguida, vou indicadar a escala em que deve-se aumentar ou diminuir o valor para ele se tornar coerente com a escala de produção

In [22]:
# ── PRODUÇÃO CONVERTIDA (usa a unidade JÁ corrigida manualmente) ────────────
# Recalcula o fator com sig_unidmed_novo (pode ter sido corrigido na verif. manual)
df_ibama_EF_und['_und_norm'] = df_ibama_EF_und['sig_unidmed_novo'].str.strip().str.upper()

df_ibama_EF_und['fatorConversao'] = df_ibama_EF_und.apply(
    lambda r: lookup_fator.get((r['cod_produto'], r['_und_norm'])),
    axis=1
)

# Produção convertida para a unidade-alvo do setor (Mg ou hl)
df_ibama_EF_und['qtd_convertida'] = (
    df_ibama_EF_und['qtd_produzida'].astype(float)
    * df_ibama_EF_und['fatorConversao'].fillna(0).astype(float)
)

df_ibama_EF_und = df_ibama_EF_und.drop(columns='_und_norm')

print("Linhas com produção convertida:", (df_ibama_EF_und['qtd_convertida'] > 0).sum())
print("Linhas zeradas (sem fator):    ", (df_ibama_EF_und['qtd_convertida'] == 0).sum())

df_producao_bruto = df_ibama_EF_und.copy()

Linhas com produção convertida: 12240
Linhas zeradas (sem fator):     8


In [23]:
serie_cols = ['CNPJ', 'MUNICIPIO', 'cod_produto']
cat_col    = 'categoria'
value_col  = 'qtd_convertida'
percentil  = 0.95

# Versão do valor SEM zeros, só para os cálculos de mediana/Q95 (0 -> NaN, ignorado)
val_calc = df_producao_bruto[value_col].replace(0, np.nan)

# Mediana da própria série (ignora zeros) e razão de cada valor vs essa mediana
mediana_serie = df_producao_bruto.assign(_v=val_calc).groupby(serie_cols)['_v'].transform('median')
df_producao_bruto['mediana_serie'] = mediana_serie
df_producao_bruto['razao_vs_mediana'] = df_producao_bruto[value_col] / mediana_serie

fora_escala = (df_producao_bruto['razao_vs_mediana'] > 10) | (df_producao_bruto['razao_vs_mediana'] < 0.1)

# Q95 da categoria (também ignora zeros)
q95_cat = df_producao_bruto.assign(_v=val_calc).groupby(cat_col)['_v'].transform(lambda x: x.quantile(percentil))
df_producao_bruto['q95_categoria'] = q95_cat
acima_q95 = df_producao_bruto[value_col] > q95_cat

# Sinaliza só quem está fora de escala E acima do Q95
mascara_outliers = (
    fora_escala
    & acima_q95
    & (df_producao_bruto['status_serie'] == 'Série Histórica Apta')
)

# NOVO: emissores consistentemente gigantes vs o setor.
# fora_escala so enxerga picos dentro da propria serie; quem e grande todo
# ano passa batido. Aqui pego a serie cuja MEDIANA supera 5x o Q95 da categoria.
mascara_gigante = (
    (mediana_serie > 5 * q95_cat)
    & (df_producao_bruto['status_serie'] == 'Série Histórica Apta')
)

df_producao_bruto['status_v04'] = np.select(
    [
        df_producao_bruto['status_serie'] == 'Histórico Insuficiente',
        df_producao_bruto[value_col] == 0,
        mascara_outliers,
        mascara_gigante,
    ],
    [
        'Histórico Insuficiente (não verificado)',
        'Sem produção convertida',
        'Escala suspeita (>10x mediana da série e > Q95)',
        'Escala suspeita (mediana da série > 5x Q95 do setor)',
    ],
    default='Escala coerente'
)

df_producao_bruto['status_v05'] = 'Fator de Escala Mantido'
df_producao_bruto['fator_escala'] = 1
df_producao_bruto['prod_temp_testes'] = df_producao_bruto['qtd_convertida']

Caso não seja coerente, vou indicar "Aplicação de favor de escala, e colocar um valor múltiplo de 10 para multiplicar (pode ser 1/10 e afins) caso valor sejam suspeito de não ser produção ou incoerente, pode-se zerar este fator de escala.

In [24]:
# Garante CNPJ como texto antes de exportar
df_producao_bruto['mv.num_cpf_cnpj'] = df_producao_bruto['mv.num_cpf_cnpj'].astype(str)

df_producao_bruto.to_excel(
    os.path.join(repo_path, 'outputs', '4_fatorEscala.xlsx'), index=False
)

# Revisar = picos intra-serie (mascara_outliers) + emissores gigantes vs setor
mascara_revisar = mascara_outliers | mascara_gigante

df_outliers_para_verificar = (
    df_producao_bruto.loc[mascara_revisar, ['mv.num_cpf_cnpj', 'cod_produto']]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Garante os dois como texto antes de exportar
df_outliers_para_verificar['mv.num_cpf_cnpj'] = df_outliers_para_verificar['mv.num_cpf_cnpj'].astype(str)
df_outliers_para_verificar['cod_produto']     = df_outliers_para_verificar['cod_produto'].astype(str)

df_outliers_para_verificar.to_excel(
    os.path.join(repo_path, 'outputs', '4A_fatorEscala.xlsx'), index=False
)

print(f"Linhas para revisao:  {mascara_revisar.sum()}")
print(f"Pares CNPJ+produto:    {len(df_outliers_para_verificar)}")
print(f"CNPJs distintos:       {df_outliers_para_verificar['mv.num_cpf_cnpj'].nunique()}")

Linhas para revisao:  180
Pares CNPJ+produto:    67
CNPJs distintos:       63


reaplicar filtro por histórico e um novo status historico dps da analise

Agrupar os dados de uma industria por ano (algumas industriar reportam mais de umd dado por ano)

In [25]:
# Importar valores analisados
df_producao_escala = pd.read_excel(os.path.join(repo_path,'inputs',
                                                'MaterialGeradoManualmente','4_fatorEscala.xlsx'),
                                   dtype={'cod_produto':'string','CNPJ':'string'})

#df_producao_escala = df_producao_escala.drop(['prod_temp_testes'], axis=1)

df_producao_escala['CNPJ'] = (
    df_producao_escala['CNPJ']
    .astype(str)
    .str.strip()
    .str.rjust(14, '0')
)

#Calcula nova versão de prodtonhl_v0 --> v1
df_producao_escala['prodtonhl_v1'] = df_producao_escala['qtd_convertida'] * df_producao_escala['fator_escala']

In [26]:
chaves = ['CNPJ', 'MUNICIPIO', 'mv.nom_pessoa', 'num_ano', 'cod_produto']
LIMIAR_RAZAO = 2.0

# Idempotencia: se a celula ja rodou antes, descarta a coluna de saida
# para nao colidir no merge ao reexecutar.
df_producao_bruto = df_producao_bruto.drop(columns=['status_agregacao'], errors='ignore')

def agrega_qtd(s):
    nz = s[s != 0]
    if len(nz) == 0:
        return 0.0
    if len(nz) == 1:
        return nz.iloc[0]
    if nz.max() / nz.min() <= LIMIAR_RAZAO:
        return nz.mean()
    return nz.sum()

def status_agreg(s):
    if len(s) == 1:
        return 'Dado não agregado'
    nz = s[s != 0]
    if len(nz) <= 1:
        return 'Agregado (apenas zeros descartados)'
    if nz.max() / nz.min() <= LIMIAR_RAZAO:
        return 'Agregado (média de valores próximos)'
    return 'Agregado com unidades discrepantes - unidade apresentada não confiável'

# Status por grupo, calculado sobre qtd_convertida (antes de colapsar)
status_por_grupo = (
    df_producao_bruto.groupby(chaves)['qtd_convertida']
    .agg(status_agreg)
    .rename('status_agregacao')
)

# Agregação principal: 'first' nas demais colunas, função custom na produção
agg_dict = {c: 'first' for c in df_producao_bruto.columns
            if c not in chaves and c != 'qtd_convertida'}
agg_dict['qtd_convertida'] = agrega_qtd

n_antes = len(df_producao_bruto)
df_producao_bruto = df_producao_bruto.groupby(chaves, as_index=False).agg(agg_dict)

# Anexa o status_agregacao
df_producao_bruto = df_producao_bruto.merge(status_por_grupo, on=chaves, how='left')

print(f"Linhas antes:  {n_antes}")
print(f"Linhas depois: {len(df_producao_bruto)}\n")
print(df_producao_bruto['status_agregacao'].value_counts())


Linhas antes:  12248
Linhas depois: 12248

status_agregacao
Dado não agregado    12248
Name: count, dtype: int64


## 5B. Verificação manual de usinas (`5_verif_usina.xlsx`)

Loop de revisão manual: os maiores emissores foram exportados para conferência
(ver `AssociarDF_AntigoNovo.ipynb`), revisados no Excel e reimportados de
`inputs/MaterialGeradoManualmente/5_verif_usina.xlsx`.

A função abaixo faz o merge com as chaves `CNPJ`, `MUNICIPIO`, `mv.nom_pessoa` e traz as
colunas `ajuste_coord`, `lat_corrigida`, `lon_corrigida` e `usina`.

**Regra principal:** se `usina == 'não'`, **toda** a produção do CNPJ (todos os anos,
municípios e produtos) é zerada — cria-se `prodtonhl_v5` a partir de `prodtonhl_v4`,
com rastreio em `status_v09`.

> Nota: como `prodtonhl_v4` só nasce dentro de `tratamento_outliers_v3`, a função é
> **definida aqui** mas **aplicada logo após o tratamento de outliers** (seção 6.2),
> antes do cálculo da emissão — que passa a usar `prodtonhl_v5`.

In [27]:
def aplicar_verif_usina(df, repo_path, col_prod='prodtonhl_v4'):
    """
    Incorpora a verificação manual de usinas (5_verif_usina.xlsx).

    Merge (chaves: CNPJ, MUNICIPIO, mv.nom_pessoa) trazendo:
    ajuste_coord, lat_corrigida, lon_corrigida, usina.

    Coordenadas: se ajuste_coord == 'sim' mantém lat_corrigida/lon_corrigida;
    caso contrário copia LATITUDE/LONGITUDE para essas colunas.

    Se usina == 'não', TODA a produção do CNPJ é zerada:
    cria `prodtonhl_v5` a partir de `col_prod` e o status `status_v09`.
    """
    chaves     = ['CNPJ', 'MUNICIPIO', 'mv.nom_pessoa']
    cols_verif = ['ajuste_coord', 'lat_corrigida', 'lon_corrigida', 'usina']

    df_verif = pd.read_excel(
        os.path.join(repo_path, 'inputs', 'MaterialGeradoManualmente', '5_verif_usina.xlsx'),
        dtype={'CNPJ': 'string'}
    )
    df_verif['CNPJ'] = (df_verif['CNPJ'].astype(str).str.strip()
                        .str.lstrip("'").str.rjust(14, '0'))

    # Coordenadas corrigidas vêm com vírgula decimal
    for c in ['lat_corrigida', 'lon_corrigida']:
        df_verif[c] = pd.to_numeric(
            df_verif[c].astype(str).str.replace(',', '.', regex=False),
            errors='coerce'
        )

    # Normalização usada SÓ nas chaves temporárias do merge e na decisão
    # (strip + upper + sem acento); não altera as colunas do df principal
    def _norm(s):
        return (s.astype(str).str.strip().str.upper()
                 .str.normalize('NFKD').str.encode('ascii', 'ignore').str.decode('ascii'))

    print(f"-> 5_verif_usina.xlsx: {len(df_verif)} linha(s) carregada(s).")
    print("   Distribuição 'usina' (planilha):")
    for val, n in _norm(df_verif['usina'].fillna('(vazio)')).value_counts().items():
        print(f"      {val:<12} {n}")
    print("   Distribuição 'ajuste_coord' (planilha):")
    for val, n in _norm(df_verif['ajuste_coord'].fillna('(vazio)')).value_counts().items():
        print(f"      {val:<12} {n}")

    chaves_tmp = [f'_k_{k}' for k in chaves]
    for k in chaves:
        df_verif[f'_k_{k}'] = _norm(df_verif[k])

    dup = df_verif.duplicated(subset=chaves_tmp)
    if dup.any():
        print(f"-> Aviso: {dup.sum()} chave(s) duplicada(s) em 5_verif_usina.xlsx — mantendo a primeira.")
        df_verif = df_verif[~dup]

    df_out = df.copy()
    for k in chaves:
        df_out[f'_k_{k}'] = _norm(df_out[k])

    df_out = df_out.merge(df_verif[chaves_tmp + cols_verif], on=chaves_tmp, how='left')
    df_out = df_out.drop(columns=chaves_tmp)

    print(f"-> Merge verif_usina: {df_out['usina'].notna().sum()} de {len(df_out)} "
          f"linhas com verificação manual ({df_out['usina'].isna().sum()} sem match).")

    # Coordenadas finais: ajuste_coord == 'sim' -> mantém lat/lon_corrigida;
    # caso contrário copia LATITUDE/LONGITUDE para lat/lon_corrigida.
    ajuste_norm = _norm(df_out['ajuste_coord'].fillna(''))
    mask_ajuste = ajuste_norm == 'SIM'
    print("-> Contagem 'ajuste_coord' (após merge):")
    print(f"      SIM (mantém corrigida)      {mask_ajuste.sum()}")
    print(f"      Demais (copia LAT/LON orig) {(~mask_ajuste).sum()}")
    df_out['lat_corrigida'] = df_out['lat_corrigida'].where(mask_ajuste, df_out['LATITUDE'])
    df_out['lon_corrigida'] = df_out['lon_corrigida'].where(mask_ajuste, df_out['LONGITUDE'])
    print(f"   -> lat_corrigida/lon_corrigida preenchidas: "
          f"{df_out['lat_corrigida'].notna().sum()}/{df_out['lon_corrigida'].notna().sum()} "
          f"de {len(df_out)} linhas.")

    # usina == 'não' -> zera TODA a produção do CNPJ (cria prodtonhl_v5)
    usina_norm = _norm(df_out['usina'].fillna(''))
    print("-> Contagem 'usina' (após merge, por linha):")
    print(f"      SIM (confirmada)     {(usina_norm == 'SIM').sum()}")
    print(f"      NAO (zera produção)  {(usina_norm == 'NAO').sum()}")
    print(f"      Outros (preenchido)  {(df_out['usina'].notna() & ~usina_norm.isin(['SIM', 'NAO'])).sum()}")
    print(f"      Sem verificação      {df_out['usina'].isna().sum()}")

    cnpjs_nao  = set(df_out.loc[usina_norm == 'NAO', 'CNPJ'])
    mask_zerar = df_out['CNPJ'].isin(cnpjs_nao)

    df_out['prodtonhl_v5'] = df_out[col_prod].where(~mask_zerar, 0)
    df_out['status_v09'] = np.select(
        [mask_zerar, usina_norm == 'SIM', df_out['usina'].notna()],
        ['Produção zerada (não é usina)', 'Usina confirmada', 'Verificação inconclusiva'],
        default='Sem verificação manual'
    )

    prod_zerada = df_out.loc[mask_zerar, col_prod].sum()
    print(f"-> {len(cnpjs_nao)} CNPJ(s) marcados como 'não usina': "
          f"{mask_zerar.sum()} linha(s) zeradas em prodtonhl_v5 "
          f"({prod_zerada:,.0f} de '{col_prod}' removidos).")
    print("-> Distribuição 'status_v09':")
    for val, n in df_out['status_v09'].value_counts().items():
        print(f"      {val:<32} {n}")

    return df_out

## 6. Análise 03: Remoção de outliers, filtro e preenchimento de série

### 6.1 Funções

Função de tratamento de outliers utilizada

In [28]:
def tratamento_outliers_v3(
    df: pd.DataFrame,
    janela_movel: int = 5,
    fator_mediana: float = 3.0,
    fator_aumento_anual: float = 2.0,
    fator_reducao_anual: float = 0.5
) -> pd.DataFrame:
    """
    Realiza um tratamento completo e rastreável de séries temporais de produção.

    Etapas:
    1. Filtragem (v2): séries com histórico insuficiente ou inteiramente zeradas são excluídas.
    2. Correção de Outliers (v3): detecção iterativa com classificação do tipo de erro
       (erro de unidade ÷1000/×1000 vs erro de digitação).
    3. Preenchimento de Lacunas (v4): anos faltantes preenchidos por interpolação linear.
    """
    print("=" * 65)
    print("  TRATAMENTO DE OUTLIERS v3 (iterativo + classificação)")
    print("=" * 65)
    print(f"  Parâmetros:")
    print(f"    janela_movel        = {janela_movel}   (anos na janela da mediana móvel)")
    print(f"    fator_mediana       = {fator_mediana}   (outlier se > {fator_mediana}× ou < 1/{fator_mediana}× mediana)")
    print(f"    fator_aumento_anual = {fator_aumento_anual}   (flag se ano > {fator_aumento_anual}× ano anterior)")
    print(f"    fator_reducao_anual = {fator_reducao_anual}   (flag se ano < {fator_reducao_anual}× ano anterior)")
    print("  Nota: com detecção iterativa, falsos positivos anuais são")
    print("  des-flagados automaticamente nas passadas seguintes.")
    print("=" * 65)

    # --- 0. PREPARAÇÃO ---
    df_processado = df.copy()
    group_cols = ['CNPJ', 'MUNICIPIO', 'cod_produto']
    year_col   = 'num_ano'
    agg_cols   = group_cols + [year_col]

    colunas_essenciais = group_cols + [year_col, 'prodtonhl_v1', 'SITUACAO CADASTRAL', 'status_serie']
    for col in colunas_essenciais:
        if col not in df_processado.columns:
            raise ValueError(f"A coluna necessária '{col}' não foi encontrada no DataFrame.")

    if df_processado.duplicated(subset=agg_cols).any():
        print("-> Consolidando registros duplicados (usando 'mean')...")
        agg_dict  = {'prodtonhl_v1': 'mean'}
        other_cols = {c: 'first' for c in df.columns if c not in agg_cols and c != 'prodtonhl_v1'}
        agg_dict.update(other_cols)
        df_processado = df_processado.groupby(agg_cols, as_index=False).agg(agg_dict)

    total_inicial_consolidado = len(df_processado)
    n_grupos = df_processado.groupby(group_cols).ngroups
    print(f"\n-> {total_inicial_consolidado} registros únicos · {n_grupos} grupos (CNPJ×município×produto)")

    df_processado['prodtonhl_v2'] = df_processado['prodtonhl_v1']
    df_processado['status_v06']   = 'Apto para Análise'
    df_processado['status_v07']   = ''
    df_processado['status_v08']   = ''

    # --- 1. FILTRAGEM (prodtonhl_v2 + status_v06) ---
    print("\n--- Etapa 1: Filtragem (cria prodtonhl_v2) ---")

    # status_serie pré-calculado antes da função (Cell 43)
    mascara_insuficiente = df_processado['status_serie'] == 'Histórico Insuficiente'
    df_processado.loc[mascara_insuficiente, 'status_v06']   = 'Histórico Insuficiente'
    df_processado.loc[mascara_insuficiente, 'prodtonhl_v2'] = 0

    df_filtrado = df_processado[~mascara_insuficiente].copy()

    if not df_filtrado.empty:
        somas_grupo   = df_filtrado.groupby(group_cols)['prodtonhl_v1'].transform('sum')
        mascara_zerada = somas_grupo == 0
        indices_zerados = df_filtrado[mascara_zerada].index
        df_processado.loc[indices_zerados, 'status_v06']   = 'Série Zerada'
        df_processado.loc[indices_zerados, 'prodtonhl_v2'] = 0
        df_filtrado = df_filtrado[~mascara_zerada]

    total_s1 = len(df_processado)
    aptos_s1 = (df_processado['status_v06'] == 'Apto para Análise').sum()
    insuf_s1 = (df_processado['status_v06'] == 'Histórico Insuficiente').sum()
    zero_s1  = (df_processado['status_v06'] == 'Série Zerada').sum()
    print(f"  Aptos p/ análise    : {aptos_s1:>6}  ({aptos_s1/total_s1*100:.1f}%)")
    print(f"  Histórico insuf.    : {insuf_s1:>6}  ({insuf_s1/total_s1*100:.1f}%)")
    print(f"  Série zerada        : {zero_s1:>6}  ({zero_s1/total_s1*100:.1f}%)")

    # --- 2. CORREÇÃO DE OUTLIERS (prodtonhl_v3 + status_v07) ---
    print("\n--- Etapa 2: Correção de outliers (cria prodtonhl_v3) ---")

    df_processado['status_v07']   = 'Não Aplicável'
    df_processado['prodtonhl_v3'] = df_processado['prodtonhl_v2']
    df_processado['flag_outlier'] = False

    if not df_filtrado.empty:
        df_filtrado = df_filtrado.sort_values(by=agg_cols)

        # Detecção iterativa: a cada passe, zeros e outliers já detectados são mascarados
        # da mediana móvel → resolve contaminação de vizinho (ex: caso Seara DF)
        max_iter    = 10
        flag_outlier = pd.Series(False, index=df_filtrado.index)

        print(f"  Detecção iterativa (max {max_iter} passadas, convergência por igualdade de flags):")
        for iter_n in range(max_iter):
            # Mascara: zeros e outliers já detectados → NaN (só para cálculo, não para correção)
            serie_calc = df_filtrado['prodtonhl_v2'].where(
                (df_filtrado['prodtonhl_v2'] > 0) & (~flag_outlier), np.nan
            )
            df_filtrado['_calc'] = serie_calc

            medianas_moveis = df_filtrado.groupby(group_cols)['_calc'].transform(
                lambda x: x.rolling(window=janela_movel, min_periods=1, center=True).median()
            )
            lim_sup = medianas_moveis * fator_mediana
            lim_inf = medianas_moveis / fator_mediana

            prod_ant = df_filtrado.groupby(group_cols)['_calc'].transform(lambda x: x.shift(1))
            razao    = serie_calc / prod_ant

            # Zeros não são outliers — já são erros confirmados do Stage 1 ou da série
            m_mediana = (
                ((df_filtrado['prodtonhl_v2'] > lim_sup) | (df_filtrado['prodtonhl_v2'] < lim_inf))
                & (medianas_moveis > 0)
                & (df_filtrado['prodtonhl_v2'] > 0)
            )
            m_anual  = (razao > fator_aumento_anual) | (razao < fator_reducao_anual)
            new_flags = (m_mediana | m_anual) & (df_filtrado['status_v06'] == 'Apto para Análise')

            adicionados = (new_flags & ~flag_outlier).sum()
            removidos   = (flag_outlier & ~new_flags).sum()
            print(f"    Passada {iter_n+1:>2}: {new_flags.sum():>4} flags  "
                  f"(+{adicionados} adicionados / -{removidos} removidos)")

            if new_flags.equals(flag_outlier):
                print(f"  ✓ Convergiu em {iter_n+1} passada(s).")
                break
            flag_outlier = new_flags

        df_filtrado.drop(columns=['_calc'], inplace=True, errors='ignore')
        df_filtrado['flag_outlier'] = flag_outlier

        n_flag = flag_outlier.sum()
        print(f"\n  Outliers detectados: {n_flag} de {len(df_filtrado)} linhas aptas "
              f"({n_flag/len(df_filtrado)*100:.1f}%)")
        if n_flag > 0:
            vals = df_filtrado.loc[flag_outlier, 'prodtonhl_v2']
            print(f"  Faixa dos outliers: min={vals.min():.2f}  "
                  f"max={vals.max():.2e}  mediana={vals.median():.2f}")

        df_filtrado['status_v07'] = 'Valor Mantido'

        def _corrigir_outliers(grupo):
            pontos_validos = grupo[~grupo['flag_outlier'] & (grupo['prodtonhl_v2'] > 0)]
            mediana_serie  = pontos_validos['prodtonhl_v2'].median()

            for idx in grupo[grupo['flag_outlier']].index:
                ano_outlier = grupo.loc[idx, year_col]
                v           = grupo.loc[idx, 'prodtonhl_v2']

                # Zeros confirmados → NaN para Stage 3 interpolar
                if v == 0:
                    grupo.loc[idx, 'prodtonhl_v3'] = np.nan
                    grupo.loc[idx, 'status_v07']   = 'Corrigido (Zero confirmado → NaN)'
                    continue

                # Testa fator ÷1000: se recoloca na faixa da série → erro de unidade
                if pd.notna(mediana_serie) and mediana_serie > 0:
                    if mediana_serie / fator_mediana <= v / 1000 <= mediana_serie * fator_mediana:
                        grupo.loc[idx, 'prodtonhl_v3'] = v / 1000
                        grupo.loc[idx, 'status_v07']   = 'Corrigido (Erro Unidade ÷1000)'
                        continue
                    # Testa fator ×1000
                    if mediana_serie / fator_mediana <= v * 1000 <= mediana_serie * fator_mediana:
                        grupo.loc[idx, 'prodtonhl_v3'] = v * 1000
                        grupo.loc[idx, 'status_v07']   = 'Corrigido (Erro Unidade ×1000)'
                        continue

                # Nenhum fator de escala encaixa → erro de digitação → interpola/extrapola/mediana
                validos_ant = pontos_validos[pontos_validos[year_col] < ano_outlier]
                validos_pos = pontos_validos[pontos_validos[year_col] > ano_outlier]
                valor_sub, metodo = np.nan, ""

                if not validos_ant.empty and not validos_pos.empty:
                    p_ant, p_pos = validos_ant.iloc[-1], validos_pos.iloc[0]
                    dy = p_pos['prodtonhl_v2'] - p_ant['prodtonhl_v2']
                    dx = p_pos[year_col] - p_ant[year_col]
                    if dx > 0:
                        valor_sub = p_ant['prodtonhl_v2'] + dy * ((ano_outlier - p_ant[year_col]) / dx)
                        metodo    = 'Corrigido (Erro Digitação → Interpolação)'
                elif len(validos_ant) >= 2:
                    p1, p2 = validos_ant.iloc[-1], validos_ant.iloc[-2]
                    dy, dx = p1['prodtonhl_v2'] - p2['prodtonhl_v2'], p1[year_col] - p2[year_col]
                    if dx > 0:
                        valor_sub = p1['prodtonhl_v2'] + (dy/dx) * (ano_outlier - p1[year_col])
                        metodo    = 'Corrigido (Erro Digitação → Extrapolação Fwd)'
                elif len(validos_pos) >= 2:
                    p1, p2 = validos_pos.iloc[0], validos_pos.iloc[1]
                    dy, dx = p2['prodtonhl_v2'] - p1['prodtonhl_v2'], p2[year_col] - p1[year_col]
                    if dx > 0:
                        valor_sub = p1['prodtonhl_v2'] + (dy/dx) * (ano_outlier - p1[year_col])
                        metodo    = 'Corrigido (Erro Digitação → Extrapolação Bwd)'
                else:
                    if pd.notna(mediana_serie):
                        valor_sub, metodo = mediana_serie, 'Corrigido (Erro Digitação → Mediana Fallback)'

                if pd.notna(valor_sub):
                    grupo.loc[idx, 'prodtonhl_v3'] = max(0, valor_sub)
                    grupo.loc[idx, 'status_v07']   = metodo
            return grupo

        df_corrigido = df_filtrado.groupby(group_cols, group_keys=False).apply(
            _corrigir_outliers, include_groups=False
        )
        df_processado.update(df_corrigido)

        # Resumo por tipo de correção
        mask_apto = df_processado['status_v06'] == 'Apto para Análise'
        contagem_corr = df_processado[mask_apto]['status_v07'].value_counts()
        print(f"\n  Correções por tipo (base: {mask_apto.sum()} linhas aptas):")
        for status, cnt in contagem_corr.items():
            pct = cnt / mask_apto.sum() * 100
            print(f"    {cnt:>5}  ({pct:>4.1f}%)  {status}")

        # Top-10 maiores correções para inspeção visual
        mask_corr = df_processado['status_v07'].str.startswith('Corrigido', na=False)
        if mask_corr.any():
            top = (
                df_processado[mask_corr][
                    ['CNPJ', 'MUNICIPIO', 'cod_produto', 'num_ano',
                     'prodtonhl_v2', 'prodtonhl_v3', 'status_v07']
                ]
                .copy()
            )
            top['razao_v2_v3'] = (top['prodtonhl_v2'] / top['prodtonhl_v3'].replace(0, np.nan)).abs()
            top = top.sort_values('razao_v2_v3', ascending=False).head(10)
            print("\n  Top-10 maiores correções (razão |v2/v3|):")
            pd.set_option('display.max_colwidth', 30)
            pd.set_option('display.float_format', '{:.2f}'.format)
            print(top.drop(columns='razao_v2_v3').to_string(index=False))
            pd.reset_option('display.max_colwidth')
            pd.reset_option('display.float_format')

    # --- 3. PREENCHIMENTO DE LACUNAS (prodtonhl_v4 + status_v08) ---
    print("\n--- Etapa 3: Preenchimento de lacunas (cria prodtonhl_v4) ---")

    df_processado['status_v08']   = 'Não Aplicável'
    df_processado['prodtonhl_v4'] = df_processado['prodtonhl_v3']

    ano_min_geral = df_processado[year_col].min()
    ano_max_geral = df_processado[year_col].max()
    print(f"  Intervalo global: {ano_min_geral} – {ano_max_geral}")

    def _preencher_grupo(grupo, ano_min_g, ano_max_g):
        status_cadastral = str(grupo['SITUACAO CADASTRAL'].iloc[0])
        num_pontos = grupo.shape[0]
        densidade  = num_pontos / (ano_max_g - ano_min_g + 1)

        grupo['status_v08'] = 'Valor Mantido'

        if status_cadastral.upper() == 'ATIVA' and densidade >= 0.75:
            intervalo = range(ano_min_g, ano_max_g + 1)
            status_preenchimento = 'Preenchido (Global - Interpolação)'
        elif status_cadastral.upper().startswith('ENCERRAD') or (
            status_cadastral.upper() == 'ATIVA' and densidade < 0.75
        ):
            intervalo = range(grupo['num_ano'].min(), grupo['num_ano'].max() + 1)
            status_preenchimento = 'Preenchido (Local - Interpolação)'
        else:
            grupo['prodtonhl_v4'] = 0
            grupo['status_v08']   = 'Zerado (Status Inválido/Outro)'
            return grupo

        grupo_reindex = grupo.set_index('num_ano').reindex(intervalo)
        mask_preenchidas = grupo_reindex['prodtonhl_v3'].isna()

        grupo_reindex['prodtonhl_v4'] = grupo_reindex['prodtonhl_v3'].interpolate(
            method='index', limit_direction='both'
        )
        grupo_reindex.loc[mask_preenchidas, 'status_v08'] = status_preenchimento

        static_cols = [c for c in grupo_reindex.columns if c not in [
            'prodtonhl_v1', 'prodtonhl_v2', 'prodtonhl_v3', 'prodtonhl_v4',
            'status_v06', 'status_v07', 'status_v08', 'flag_outlier'
        ]]
        grupo_reindex[static_cols] = grupo_reindex[static_cols].ffill().bfill().infer_objects(copy=False)

        return grupo_reindex.reset_index()

    df_aptos = df_processado[df_processado['status_v06'] == 'Apto para Análise']
    if not df_aptos.empty:
        lista_grupos = []
        for _, grupo in df_aptos.groupby(group_cols):
            lista_grupos.append(_preencher_grupo(grupo, ano_min_geral, ano_max_geral))

        if lista_grupos:
            df_final_preenchido = pd.concat(lista_grupos, ignore_index=True)
            df_nao_proc = df_processado[df_processado['status_v06'] != 'Apto para Análise'].copy()
            df_processado = pd.concat([df_final_preenchido, df_nao_proc], ignore_index=True)

    # Novas linhas (de preenchimento) não têm flag_outlier → padrão False
    if 'flag_outlier' in df_processado.columns:
        df_processado['flag_outlier'] = df_processado['flag_outlier'].fillna(False)

    total_s3  = len(df_processado)
    mantidos  = (df_processado['status_v08'] == 'Valor Mantido').sum()
    global_p  = (df_processado['status_v08'] == 'Preenchido (Global - Interpolação)').sum()
    local_p   = (df_processado['status_v08'] == 'Preenchido (Local - Interpolação)').sum()
    zerados   = (df_processado['status_v08'] == 'Zerado (Status Inválido/Outro)').sum()
    print(f"  Mantidos originais  : {mantidos:>6}")
    print(f"  Preenchidos global  : {global_p:>6}")
    print(f"  Preenchidos local   : {local_p:>6}")
    print(f"  Zerados (inv. cad.) : {zerados:>6}")
    print(f"  Total linhas saída  : {total_s3:>6}")

    # --- RESUMO GERAL ---
    qtd_mantidos_integrais = (
        (df_processado['status_v06'] == 'Apto para Análise') &
        (df_processado['status_v07'] == 'Valor Mantido') &
        (df_processado['status_v08'] == 'Valor Mantido')
    ).sum()
    qtd_corrigidos  = df_processado['status_v07'].str.startswith('Corrigido', na=False).sum()
    qtd_preenchidos = df_processado['status_v08'].str.contains('Preenchido', na=False).sum()

    print("\n" + "=" * 65)
    print("  RESUMO GERAL")
    print("=" * 65)
    print(f"  Dados iniciais (consolidados)  : {total_inicial_consolidado:>6}")
    print(f"  ├─ Filtrados (Etapa 1)         : {insuf_s1 + zero_s1:>6}  (insuf.={insuf_s1}, zerada={zero_s1})")
    print(f"  ├─ Mantidos sem alteração      : {qtd_mantidos_integrais:>6}")
    print(f"  └─ Corrigidos (outliers)       : {qtd_corrigidos:>6}  ({qtd_corrigidos/total_inicial_consolidado*100:.1f}%)")
    print(f"  Linhas adicionadas (preenchi.) : {qtd_preenchidos:>6}")
    print(f"  Total saída                    : {total_s3:>6}")
    print("=" * 65)

    # flag_outlier incluída na saída para auditoria
    # (True = ponto detectado como outlier na Etapa 2)
    colunas_finais = [c for c in df.columns if c not in ['prodtonhl_v1', 'flag_outlier']] + \
                     ['prodtonhl_v1', 'prodtonhl_v2', 'prodtonhl_v3', 'prodtonhl_v4',
                      'status_v06', 'status_v07', 'status_v08', 'flag_outlier']

    colunas_existentes = [col for col in colunas_finais if col in df_processado.columns]
    return df_processado[colunas_existentes].sort_values(by=agg_cols).reset_index(drop=True)


### 6.2 Código

In [29]:
#Como ajustei a classificação dos produtos posteriormente, mas o material
# foi feito de maneira manual, devo arrumar aqui

df_producao_final = tratamento_outliers_v3(df_producao_escala)

# Verificação manual de usinas (seção 5B): usina == 'não' -> zera produção do CNPJ (v4 -> v5)
df_producao_final = aplicar_verif_usina(df_producao_final, repo_path, col_prod='prodtonhl_v4')

df_inventario = df_producao_final.copy()
df_inventario['Emissão NMCOV (kg)'] = (df_inventario['prodtonhl_v5'] * df_inventario['Value'].astype(float))
df_inventario['Emissão NMCOV CI_lower (kg)'] = (df_inventario['prodtonhl_v5'] * df_inventario['CI_lower'].astype(float))
df_inventario['Emissão NMCOV CI_upper (kg)'] = (df_inventario['prodtonhl_v5'] * df_inventario['CI_upper'].astype(float))

# Como edgar é em ton, vou deixar tudo em toneladas
df_inventario['Emissão NMCOV (ton)'] = df_inventario['Emissão NMCOV (kg)']/1000
df_inventario['Emissão NMCOV CI_lower (ton)'] = df_inventario['Emissão NMCOV CI_lower (kg)']/1000
df_inventario['Emissão NMCOV CI_upper (ton)'] = df_inventario['Emissão NMCOV CI_upper (kg)']/1000

  TRATAMENTO DE OUTLIERS v3 (iterativo + classificação)
  Parâmetros:
    janela_movel        = 5   (anos na janela da mediana móvel)
    fator_mediana       = 3.0   (outlier se > 3.0× ou < 1/3.0× mediana)
    fator_aumento_anual = 2.0   (flag se ano > 2.0× ano anterior)
    fator_reducao_anual = 0.5   (flag se ano < 0.5× ano anterior)
  Nota: com detecção iterativa, falsos positivos anuais são
  des-flagados automaticamente nas passadas seguintes.
-> Consolidando registros duplicados (usando 'mean')...

-> 14858 registros únicos · 3777 grupos (CNPJ×município×produto)

--- Etapa 1: Filtragem (cria prodtonhl_v2) ---
  Aptos p/ análise    :  11894  (80.1%)
  Histórico insuf.    :   2952  (19.9%)
  Série zerada        :     12  (0.1%)

--- Etapa 2: Correção de outliers (cria prodtonhl_v3) ---
  Detecção iterativa (max 10 passadas, convergência por igualdade de flags):
    Passada  1: 2002 flags  (+2002 adicionados / -0 removidos)
    Passada  2:  815 flags  (+20 adicionados / -1207 removi

C:\Users\glima\AppData\Local\Temp\ipykernel_14712\3775794859.py:279: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  grupo_reindex[static_cols] = grupo_reindex[static_cols].ffill().bfill().infer_objects(copy=False)
C:\Users\glima\AppData\Local\Temp\ipykernel_14712\3775794859.py:279: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  grupo_reindex[static_cols] = grupo_reindex[static_cols].ffill().bfill().infer_objects(copy=False)
C:\Users\glima\AppData\Local\Temp\ipykernel_14712\3775794859.py:279: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version.

  Mantidos originais  :  11879
  Preenchidos global  :    342
  Preenchidos local   :    212
  Zerados (inv. cad.) :     15
  Total linhas saída  :  15412

  RESUMO GERAL
  Dados iniciais (consolidados)  :  14858
  ├─ Filtrados (Etapa 1)         :   2964  (insuf.=2952, zerada=12)
  ├─ Mantidos sem alteração      :  10997
  └─ Corrigidos (outliers)       :    883  (5.9%)
  Linhas adicionadas (preenchi.) :    554
  Total saída                    :  15412
-> 5_verif_usina.xlsx: 1735 linha(s) carregada(s).
   Distribuição 'usina' (planilha):
      (VAZIO)      832
      SIM          816
      NAO          87
   Distribuição 'ajuste_coord' (planilha):
      NAO          804
      SIM          484
      (VAZIO)      447
-> Merge verif_usina: 7486 de 15412 linhas com verificação manual (7926 sem match).
-> Contagem 'ajuste_coord' (após merge):
      SIM (mantém corrigida)      3616
      Demais (copia LAT/LON orig) 11796
   -> lat_corrigida/lon_corrigida preenchidas: 15341/15337 de 15412 linh

In [30]:
#Como ajustei a classificação dos produtos posteriormente, mas o material
# foi feito de maneira manual, devo arrumar aqui

df_producao_final = tratamento_outliers_v3(df_producao_escala)

# Verificação manual de usinas (seção 5B): usina == 'não' -> zera produção do CNPJ (v1 -> v5)
df_producao_final = aplicar_verif_usina(df_producao_final, repo_path, col_prod='prodtonhl_v1')

df_inventario = df_producao_final.copy()
df_inventario['Emissão NMCOV (kg)'] = (df_inventario['prodtonhl_v5'] * df_inventario['Value'].astype(float))
df_inventario['Emissão NMCOV CI_lower (kg)'] = (df_inventario['prodtonhl_v5'] * df_inventario['CI_lower'].astype(float))
df_inventario['Emissão NMCOV CI_upper (kg)'] = (df_inventario['prodtonhl_v5'] * df_inventario['CI_upper'].astype(float))

# Como edgar é em ton, vou deixar tudo em toneladas
df_inventario['Emissão NMCOV (ton)'] = df_inventario['Emissão NMCOV (kg)']/1000
df_inventario['Emissão NMCOV CI_lower (ton)'] = df_inventario['Emissão NMCOV CI_lower (kg)']/1000
df_inventario['Emissão NMCOV CI_upper (ton)'] = df_inventario['Emissão NMCOV CI_upper (kg)']/1000

  TRATAMENTO DE OUTLIERS v3 (iterativo + classificação)
  Parâmetros:
    janela_movel        = 5   (anos na janela da mediana móvel)
    fator_mediana       = 3.0   (outlier se > 3.0× ou < 1/3.0× mediana)
    fator_aumento_anual = 2.0   (flag se ano > 2.0× ano anterior)
    fator_reducao_anual = 0.5   (flag se ano < 0.5× ano anterior)
  Nota: com detecção iterativa, falsos positivos anuais são
  des-flagados automaticamente nas passadas seguintes.
-> Consolidando registros duplicados (usando 'mean')...

-> 14858 registros únicos · 3777 grupos (CNPJ×município×produto)

--- Etapa 1: Filtragem (cria prodtonhl_v2) ---
  Aptos p/ análise    :  11894  (80.1%)
  Histórico insuf.    :   2952  (19.9%)
  Série zerada        :     12  (0.1%)

--- Etapa 2: Correção de outliers (cria prodtonhl_v3) ---
  Detecção iterativa (max 10 passadas, convergência por igualdade de flags):
    Passada  1: 2002 flags  (+2002 adicionados / -0 removidos)
    Passada  2:  815 flags  (+20 adicionados / -1207 removi

C:\Users\glima\AppData\Local\Temp\ipykernel_14712\3775794859.py:279: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  grupo_reindex[static_cols] = grupo_reindex[static_cols].ffill().bfill().infer_objects(copy=False)
C:\Users\glima\AppData\Local\Temp\ipykernel_14712\3775794859.py:279: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  grupo_reindex[static_cols] = grupo_reindex[static_cols].ffill().bfill().infer_objects(copy=False)
C:\Users\glima\AppData\Local\Temp\ipykernel_14712\3775794859.py:279: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version.

  Mantidos originais  :  11879
  Preenchidos global  :    342
  Preenchidos local   :    212
  Zerados (inv. cad.) :     15
  Total linhas saída  :  15412

  RESUMO GERAL
  Dados iniciais (consolidados)  :  14858
  ├─ Filtrados (Etapa 1)         :   2964  (insuf.=2952, zerada=12)
  ├─ Mantidos sem alteração      :  10997
  └─ Corrigidos (outliers)       :    883  (5.9%)
  Linhas adicionadas (preenchi.) :    554
  Total saída                    :  15412
-> 5_verif_usina.xlsx: 1735 linha(s) carregada(s).
   Distribuição 'usina' (planilha):
      (VAZIO)      832
      SIM          816
      NAO          87
   Distribuição 'ajuste_coord' (planilha):
      NAO          804
      SIM          484
      (VAZIO)      447
-> Merge verif_usina: 7486 de 15412 linhas com verificação manual (7926 sem match).
-> Contagem 'ajuste_coord' (após merge):
      SIM (mantém corrigida)      3616
      Demais (copia LAT/LON orig) 11796
   -> lat_corrigida/lon_corrigida preenchidas: 15341/15337 de 15412 linh

## 7. Exportar para realizar análises em outro código

In [31]:
df_inventario['CNPJ'] = "'" + df_inventario['CNPJ'].astype(str)
df_inventario = df_inventario[df_inventario['prodtonhl_v4'] != 0].copy()
df_inventario['LATITUDE'] = (
    df_inventario['LATITUDE']
    .astype(str)
    .str.replace(',', '.', regex=False)
    .astype(float)
)

df_inventario['LONGITUDE'] = (
    df_inventario['LONGITUDE']
    .astype(str)
    .str.replace(',', '.', regex=False)
    .astype(float)
)

df_inventario.to_csv(
    os.path.join(repo_path,'outputs','inventarioEmissoesIndustriaisIndustriaAlimenticiaBR_V3.csv'),
    index=False, encoding='utf-8-sig'
)


Ao final, criar um read-me e estudar sobre o CLAUDE.md, que explica ao claude as diretrizes do processo, documentando-o